# Forensics: what does the shop's hidden model reward? (H1–H11)

Questions from the plan (`docs/memory/06_plan_ganador.md` §3). All logic lives in `rtrec/data/forensics.py`;
this notebook only runs it and shows the tables, so **it reruns unchanged on the real dump**:

```bash
python scripts/download_history.py --out data --live      # real data (needs the key)
python -m sim.dump --out data --live-minutes 120          # or synthetic data
RTREC_DATA=data jupyter nbconvert --to notebook --execute notebooks/forensics.ipynb
```

Click rates are compared **per unit of examination** (clicks / Σ examine[slot], with the position curve fitted in H2), so a
feature is not credited for the slot the old carousel happened to put it in. Results summary: `docs/FORENSICS.md`.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from rtrec.data.load import load_history, load_live
from rtrec.data import forensics as FZ
pd.set_option("display.width", 160, "display.max_columns", 20)

DATA = pathlib.Path(os.environ.get("RTREC_DATA", ROOT / "data"))
h = load_history(DATA)
live = load_live(DATA)
print("source:", h.source)
print(f"{len(h.catalog)} products, {len(h.sessions)} impressions, {len(h.events)} events;",
      "live tables:", {k: len(v) for k, v in live.items()})

In [ ]:
fz = FZ.Forensics(h, max_sessions=400_000)   # samples impressions if the dump is bigger
def show(f):
    print(f.question); print("->", f.answer)
    return f.table

## H1
Is it worth showing what the shopper just viewed? Decides whether viewed items lead the list.

In [ ]:
show(fz.h1_viewed_products())

## H2
Position curve: raw clicks per slot vs the fitted examination probability (product effects removed).

In [ ]:
show(fz.h2_position_bias())

## H3
Session similarity features: same subcategory / category / brand as the views, price ratio.

In [ ]:
show(fz.h3_what_drives_clicks())

## H4
Price sensitivity by device and region (logit slope of P(buy | cart) on log price).

In [ ]:
show(fz.h4_price_sensitivity())

## H5
Long-term history: earlier-session affinity, and the repeat-purchase penalty.

In [ ]:
show(fz.h5_user_history())

## H6
Do subcategory view surges in organic events predict carousel clicks, and how long does it last?

In [ ]:
show(fz.h6_trends())

## H7
Price drops and raises (needs live catalog_updates).

In [ ]:
show(fz.h7_price_changes(live.get('events'), live.get('catalog_updates')))

## H8
Shape of the target: one click / purchase per impression, purchase implies click.

In [ ]:
show(fz.h8_target_shape())

## H9
Structure of the logged propensities (decides the off-policy estimator).

In [ ]:
show(fz.h9_propensity())

## H10, H11 (live tables)
Requests mix and stock-out risk; need `data/live/` (download with `--live`, or `sim.dump --live-minutes`).

In [ ]:
if "requests" in live:
    display(show(FZ.h10_live_requests(live["requests"])))
if "catalog_updates" in live:
    top = h.events[h.events.event_type == "purchase"].product_id.value_counts().index[:100].tolist()
    display(show(FZ.h11_stock_risk(live["catalog_updates"], top, live.get("feedback"))))